In [2]:
import pandas as pd
import re

# Load your CSV file
df = pd.read_csv("Earthquake.csv")

# Split multiple countries into separate records
df["country"] = df["country"].astype(str).str.split(",")
df = df.explode("country")
df["country"] = df["country"].str.strip()

# Add suffix to disaster_id for multiple countries
df["disaster_id"] = df.groupby("disaster_id").cumcount().add(1).astype(str).radd(df["disaster_id"] + "_")
df["disaster_id"] = df["disaster_id"].replace(r'_1$', '', regex=True)

# Function to clean htmldescription and retain disaster color/type
def clean_description(row):
    desc = str(row["htmldescription"])
    
    # Keep only the part before "from:"
    desc = re.sub(r"from:.*", "", desc, flags=re.IGNORECASE).strip()
    
    # Identify the disaster type (e.g., Green Drought, Orange Drought, Red Flood, etc.)
    match = re.search(r"(Green|Orange|Red|Yellow|Blue)\s+\w+", desc)
    disaster_type = match.group(0) if match else "Disaster"
    
    # Create clean single-country description
    return f"{disaster_type} in {row['country']}"

# Apply the function
df["htmldescription"] = df.apply(clean_description, axis=1)

# Save the cleaned dataset
df.to_csv("cleaned_disaster_dataset.csv", index=False, encoding='utf-8-sig')

print("✅ Dataset cleaned successfully and saved as 'cleaned_Earthquake.csv'")


✅ Dataset cleaned successfully and saved as 'cleaned_Earthquake.csv'


In [5]:
import pandas as pd
import re

# Read your dataset
df = pd.read_csv("cleaned_earthquake_final.csv")  # or pd.read_csv("your_file.csv")

# --- Fix 1: Replace blank or NaN in population_estimate with 'few'
df['population_estimate'] = df['population_estimate'].replace('', 'few').fillna('few')

# --- Fix 2: Fill missing impact_zone by extracting from population_estimate text
def extract_zone(text):
    if pd.isna(text):
        return None
    text = str(text)
    mmi_match = re.search(r'MMI\s*≥?\s*[IVX]+', text)
    within_match = re.search(r'within\s*\d+\s*km', text)
    if mmi_match:
        return mmi_match.group()
    elif within_match:
        return within_match.group()
    return None

df['impact_zone'] = df.apply(
    lambda x: x['impact_zone'] if pd.notna(x['impact_zone']) and x['impact_zone'] != '' else extract_zone(x['population_estimate']),
    axis=1
)

# Save updated dataset
df.to_csv("fixed_dataset.csv", index=False)


In [6]:
import pandas as pd
import re

# Read your dataset
df = pd.read_csv("fixed_dataset.csv")  # or pd.read_csv("your_file.csv")

# --- Fix 1: Replace blank or NaN in population_estimate with 'few'
df['population_estimate'] = df['population_estimate'].replace('', 'few').fillna('few')

# --- Fix 2: Extract impact_zone from population_estimate when missing
def extract_zone(text):
    if pd.isna(text):
        return None
    text = str(text)
    
    # Match patterns like "MMI>=VII", "(in MMI>=VII)", "in MMI VII", etc.
    mmi_match = re.search(r'MMI\s*≥?\s*[IVX]+', text)
    if mmi_match:
        return mmi_match.group().replace(" ", "")
    
    # Match patterns like "within 100km" or "within 50 km"
    within_match = re.search(r'within\s*\d+\s*km', text)
    if within_match:
        return within_match.group().replace(" ", "")
    
    return None

# Apply extraction only when impact_zone is blank or NaN
df['impact_zone'] = df.apply(
    lambda x: x['impact_zone'] if pd.notna(x['impact_zone']) and x['impact_zone'] != '' 
    else extract_zone(x['population_estimate']),
    axis=1
)

# Save the updated dataset
df.to_csv("fixed_dataset_earthquake.csv", index=False)


In [7]:
import pandas as pd
import re

# Load CSV file
df = pd.read_csv("fixed_dataset_earthquake.csv")

# --- Fix 1: Replace blank or NaN in population_estimate with 'few'
df['population_estimate'] = df['population_estimate'].replace('', 'few').fillna('few')

# --- Fix 2: Extract and clean impact_zone properly ---
def extract_zone(text):
    if pd.isna(text):
        return None
    text = str(text)

    # Clean extra spaces and lowercase
    text_clean = text.lower().strip()

    # 🔹 Match MMI patterns inside or outside brackets
    mmi_pattern = re.search(r'\(?in\s*mmi\s*[≥>=]*\s*[ivx]+\)?', text_clean)
    if mmi_pattern:
        extracted = mmi_pattern.group(0)
        # Remove parentheses and "in", standardize
        cleaned = re.sub(r'[\(\)]', '', extracted)
        cleaned = cleaned.replace('in', '').replace('mmi', 'MMI').replace(' ', '')
        return cleaned.strip()

    # 🔹 Match MMI patterns without "in" (like "MMI VII" or "MMI>=VII")
    mmi_pattern2 = re.search(r'mmi\s*[≥>=]*\s*[ivx]+', text_clean)
    if mmi_pattern2:
        cleaned = mmi_pattern2.group(0).replace('mmi', 'MMI').replace(' ', '')
        return cleaned.strip()

    # 🔹 Match “within 100km” or similar patterns
    within_pattern = re.search(r'within\s*\d+\s*km', text_clean)
    if within_pattern:
        cleaned = within_pattern.group(0).replace(' ', '')
        return cleaned.strip()

    return None


# Apply extraction only when impact_zone is blank or NaN
df['impact_zone'] = df.apply(
    lambda x: x['impact_zone'] if pd.notna(x['impact_zone']) and x['impact_zone'] != '' 
    else extract_zone(x['population_estimate']),
    axis=1
)

# Save to new CSV
df.to_csv("cleaned_drought_fixed.csv", index=False, encoding='utf-8-sig')

print("✅ File saved successfully as 'cleaned_earthquake.csv'")


✅ File saved successfully as 'cleaned_earthquake.csv'


In [8]:
import pandas as pd
import re
import unicodedata

# Load CSV file (update filename if different)
fn_in = "cleaned_drought_fixed.csv"
fn_out = "cleaned_drought_fixed_mmi.csv"
df = pd.read_csv(fn_in, dtype=str)  # load as strings to avoid surprises

# Ensure the column names exist
pop_col = 'population_estimate'      # the raw text column or where text like "90 thousand (in MMI>=VII)" lives
impact_col = 'impact_zone'          # target column to fill

# If names differ, adjust the variables above accordingly.

def normalize_text(s):
    if pd.isna(s):
        return ""
    # Normalize Unicode forms and convert to a single spacing form
    s = unicodedata.normalize('NFKC', str(s))
    # Replace various Unicode 'greater or equal' with ascii '>='
    s = s.replace('≥', '>=').replace('\u2265', '>=')
    # Replace weird spaces with normal space
    s = re.sub(r'[\u00A0\u2000-\u200A\u202F\u205F\u3000]', ' ', s)
    # Collapse multiple spaces
    s = re.sub(r'\s+', ' ', s).strip()
    return s

# Robust extractor for MMI patterns
mmi_regex = re.compile(
    r'''
    [\(\[]*                       # optional opening bracket
    \s*(?:in\s*)?                 # optional leading "in"
    (mmi)                         # literal MMI (capture)
    [\s\)\]\:;,\-]*               # optional separators/spaces
    ([>&=]{0,2}|>=|&>=|≥|>=|[><]=?)? # optional operator like >=, &>= etc.
    [\s\)\]\:;,\-]*               # optional separators
    ([ivxlcdm]+)                  # roman numerals (one or more) e.g., iv, vii
    [\)\]]*                       # optional closing bracket
    ''',
    flags=re.IGNORECASE | re.VERBOSE
)

def extract_mmi_from_text(txt):
    s = normalize_text(txt)
    # quick fail
    if 'mmi' not in s.lower():
        return None

    # search for MMI patterns
    m = mmi_regex.search(s)
    if not m:
        # fallback: try looser search (find 'mmi' then up to 6 following chars containing >= or roman)
        loose = re.search(r'mmi[^a-zA-Z0-9]{0,3}[^a-zA-Z0-9]*([>=\u2265&]*\s*[ivxlcdm]+)', s, flags=re.I)
        if loose:
            fragment = loose.group(0)
        else:
            return None
    else:
        # groups: (mmi, operator, roman)
        mmi_word = m.group(1) or 'MMI'
        op = m.group(2) or ''
        roman = m.group(3) or ''
        # cleanup
        op = op.replace(' ', '')
        # convert any unicode ≥ to >= (already normalized)
        op = op.replace('≥', '>=').replace('&', '&')
        roman = roman.upper()
        # make final string: keep operator (if present) joined directly
        if op:
            return f"MMI{op}{roman}"
        else:
            # insert space between MMI and roman for readability (you requested MMI>=VII format; for no operator, use 'MMI IV' or 'MMIIV'?
            # We'll return 'MMI IV' to be readable.
            return f"MMI {roman}"

    # If we reached fallback fragment, tidy it
    frag = normalize_text(fragment)
    # clean non-alphanums except >= and & and letters
    frag = frag.replace('≥', '>=')
    frag = re.sub(r'[^A-Za-z0-9>=&]+', '', frag)
    frag = frag.upper()
    # ensure it starts with MMI
    if not frag.startswith('MMI'):
        frag = 'MMI' + frag
    return frag

# Apply extraction where impact_zone is empty or NaN
# Ensure the impact column exists
if impact_col not in df.columns:
    df[impact_col] = ""

# Apply extraction
new_impact = []
for idx, row in df.iterrows():
    current = row.get(impact_col, "")
    # If it's already present and non-empty, keep it
    if isinstance(current, str) and current.strip():
        new_impact.append(current)
        continue
    # else attempt extraction from the population_estimate or Exposed Population raw column if that exists
    text_source = ""
    if pop_col in df.columns:
        text_source = row.get(pop_col, "")
    else:
        # Try common alternate column names
        for alt in ['Exposed Population', 'exposed_population', 'exposed population']:
            if alt in df.columns:
                text_source = row.get(alt, "")
                break

    extracted = extract_mmi_from_text(text_source)
    if extracted:
        # Normalize formatting: remove spaces around >= and ensure uppercase as requested
        extracted = extracted.replace(' ', '') if '>=' in extracted else extracted  # keep space if no operator
        # But user asked earlier they'd prefer MMI>=VII; for cases with no operator we'll keep "MMI IV"
        # Let's reformat: if '>=' in string keep it, else ensure 'MMI IV' (with space)
        if '>=' in extracted:
            # ensure it's like MMI>=VII
            extracted = extracted.replace(' ', '')
        else:
            # ensure format 'MMI IV'
            if not re.match(r'MMI\s', extracted):
                # insert space before roman numerals
                extracted = re.sub(r'(MMI)([IVXLCDM]+)', r'\1 \2', extracted)
        new_impact.append(extracted)
    else:
        new_impact.append(current)  # leave as-is (blank or existing)

df[impact_col] = new_impact

# Optional: if you want to set 'few' for population_estimate blanks (you said earlier), do it:
if pop_col in df.columns:
    df[pop_col] = df[pop_col].replace('', pd.NA).fillna('few')

# Save to CSV
df.to_csv(fn_out, index=False, encoding='utf-8-sig')

# Quick test print on rows that still have empty impact_zone to inspect
empty_impact_rows = df[df[impact_col].isna() | (df[impact_col].astype(str).str.strip() == "")]
if not empty_impact_rows.empty:
    print("Rows still missing impact_zone (showing sample):")
    print(empty_impact_rows[[pop_col, impact_col]].head(10))
else:
    print("All impact_zone values filled where possible. Saved to:", fn_out)


Rows still missing impact_zone (showing sample):
   population_estimate impact_zone
5              90000.0         NaN
6                  few         NaN
9               3000.0         NaN
10              3000.0         NaN
11              2000.0         NaN
12              2000.0         NaN
14            140000.0         NaN
17            140000.0         NaN
19             40000.0         NaN
31            190000.0         NaN


In [1]:
import pandas as pd
import re

# Load your CSV file
df = pd.read_csv("cleaned_drought_fixed.csv")

def extract_exposed_population(value):
    if pd.isna(value):
        return pd.Series({"exposed_population_clean": "", "impact_zone": ""})

    # Look for text in parentheses like (in MMI>=VII)
    match_in_brackets = re.search(r"\(in\s*([^)]+)\)", value, flags=re.IGNORECASE)
    if match_in_brackets:
        impact_zone = match_in_brackets.group(1).strip()
    else:
        # Look for text after 'in ' if not in parentheses
        match_after_in = re.search(r"\bin\s+([A-Za-z0-9>=\-\s]+)", value, flags=re.IGNORECASE)
        impact_zone = match_after_in.group(1).strip() if match_after_in else ""

    # Remove "(in ...)" or "in ..." from the text for clean population value
    exposed_population_clean = re.sub(r"\(in\s*[A-Za-z0-9>=\-\s]+\)|\bin\s+[A-Za-z0-9>=\-\s]+", "", value, flags=re.IGNORECASE).strip()

    return pd.Series({
        "exposed_population_clean": exposed_population_clean,
        "impact_zone": impact_zone
    })

# Apply the extraction function to the column
df[["exposed_population_clean", "impact_zone"]] = df["Exposed Population"].apply(extract_exposed_population)

# Save to new CSV
df.to_csv("output_fixed.csv", index=False)

print("✅ Extraction complete. Check 'output_fixed.csv'")


✅ Extraction complete. Check 'output_fixed.csv'


In [2]:
import pandas as pd
import re

# Load your CSV
df = pd.read_csv("cleaned_drought_fixed.csv")

def extract_impact_zone(value):
    if pd.isna(value):
        return ""
    
    # 1️⃣ Try to find pattern inside parentheses like (in MMI>=VII)
    match_in_brackets = re.search(r"\(in\s*([^)]+)\)", value, flags=re.IGNORECASE)
    if match_in_brackets:
        return match_in_brackets.group(1).strip()

    # 2️⃣ Otherwise, look for text like "in MMI IV" or "in MMI>=VII"
    match_after_in = re.search(r"\bin\s+([A-Za-z0-9>=\-\s]+)", value, flags=re.IGNORECASE)
    if match_after_in:
        return match_after_in.group(1).strip()
    
    return ""

# Only fill impact_zone if it's currently empty or null
df["impact_zone"] = df.apply(
    lambda row: row["impact_zone"] if pd.notna(row["impact_zone"]) and str(row["impact_zone"]).strip() != "" 
    else extract_impact_zone(str(row["Exposed Population"])), 
    axis=1
)

# Save the updated CSV
df.to_csv("output_updated.csv", index=False)

print("✅ impact_zone updated successfully and saved to output_updated.csv")


✅ impact_zone updated successfully and saved to output_updated.csv


In [3]:
import pandas as pd
import re

# Load CSV
df = pd.read_csv("output_updated.csv")

def extract_impact_zone(value):
    if pd.isna(value):
        return ""
    
    match_in_brackets = re.search(r"\(in\s*([^)]+)\)", value, flags=re.IGNORECASE)
    if match_in_brackets:
        return match_in_brackets.group(1).strip()

    match_after_in = re.search(r"\bin\s+([A-Za-z0-9>=\-\s]+)", value, flags=re.IGNORECASE)
    if match_after_in:
        return match_after_in.group(1).strip()
    
    return ""

# ✅ Update impact_zone (as before)
df["impact_zone"] = df.apply(
    lambda row: row["impact_zone"] if pd.notna(row["impact_zone"]) and str(row["impact_zone"]).strip() != "" 
    else extract_impact_zone(str(row["Exposed Population"])), 
    axis=1
)

# ✅ Update population_estimate only if Exposed Population has "few"
df["population_estimate"] = df.apply(
    lambda row: "few" if (
        (pd.isna(row["population_estimate"]) or str(row["population_estimate"]).strip() == "") 
        and isinstance(row["Exposed Population"], str) 
        and re.search(r"\bfew\b", row["Exposed Population"], flags=re.IGNORECASE)
    ) else row["population_estimate"],
    axis=1
)

# Save the final version
df.to_csv("output_final.csv", index=False)

print("✅ impact_zone and population_estimate updated successfully and saved to output_final.csv")


✅ impact_zone and population_estimate updated successfully and saved to output_final.csv


In [4]:
import pandas as pd
import re

# Load your CSV
df = pd.read_csv("output_final.csv")

def clean_htmldescription(text):
    if pd.isna(text):
        return text
    
    text = str(text).strip()
    
    # Replace "M" with "Earthquake"
    text = re.sub(r"\bM\b", "Earthquake", text)
    
    # Remove "in nan", "in NaN", or "in None" (case-insensitive)
    text = re.sub(r"\bin\s*(nan|NaN|none|null)\b", "", text, flags=re.IGNORECASE)
    
    # Clean double spaces after removals
    text = re.sub(r"\s+", " ", text).strip()
    
    # Ensure proper capitalization (optional)
    text = text[0].upper() + text[1:]
    
    return text

# Apply the cleaning
df["htmldescription"] = df["htmldescription"].apply(clean_htmldescription)

# Save cleaned version
df.to_csv("cleaned_htmldescription.csv", index=False)

print("✅ htmldescription column cleaned successfully and saved to cleaned_htmldescription.csv")


✅ htmldescription column cleaned successfully and saved to cleaned_htmldescription.csv


In [5]:
import pandas as pd
import re

# Load your CSV
df = pd.read_csv("cleaned_htmldescription.csv")

df['impact_category'] = df['impact_zone'].replace({
    '100km': 'Within 100 km',
    'MMI I': 'Very low intensity',
    'MMI V': 'Moderate intensity',
    'MMI≥VII': 'High intensity',
    'MMI>=VII': 'High intensity'
})

df.to_csv("cleaned_earthquake.csv", index=False)

print("✅ htmldescription column cleaned successfully and saved to cleaned_htmldescription.csv")

✅ htmldescription column cleaned successfully and saved to cleaned_htmldescription.csv


In [6]:
import pandas as pd
import re

# Load your CSV
df = pd.read_csv("cleaned_earthquake.csv")

check = df['impact_category'].unique()

print(check)


['Within 100 km' 'Moderate intensity' 'High intensity'
 'Very low intensity' nan 'MMI  1010000' 'MMI  1230000' 'MMI  15590000'
 'MMI  300000' 'MMI  240000' 'MMI  VIII1250000' 'MMI  510000'
 'MMI  330000' 'MMI  9570000' 'MMI  16020000' 'MMI  3270000' 'MMI  930000'
 'MMI  2820000' 'MMI  10060000' 'MMI  ABOUT' 'MMI  3190000' 'MMI  1990000'
 'MMI  640' 'MMI  7690000' 'MMI  810000' 'MMI  460000' 'MMI  320000'
 'MMI  260000' 'MMI  1070000' 'MMI  230000' 'MMI  6500000' 'MMI  3630000'
 'MMI  640000' 'MMI  VIIABOUT' 'MMI  VI170000' 'MMI  1800000'
 'MMI  1670000' 'MMI  1180000' 'MMI  410000' 'MMI  VII1170000'
 'MMI  1350000' 'MMI  2630000' 'MMI  1770000' 'MMI  3930000' 'MMI  590000'
 'MMI  5120000' 'MMI  VII1640000' 'MMI  190000' 'MMI  VI180000'
 'MMI  VII180000' 'MMI  V300000' 'MMI  VABOUT' 'MMI  VIII180000'
 'MMI  3670000' 'MMI  VII2390000' 'MMI  1190000' 'MMI  1380000'
 'MMI  1340000' 'MMI  780000' 'MMI  VII1110000' 'MMI  VI390000'
 'MMI  390000' 'MMI  VIII390000' 'MMI  VII1770000' 'MMI  5160

In [9]:
import pandas as pd
import re

# Load your CSV
df = pd.read_csv("cleaned_earthquake.csv")

def categorize_impact(value):
    if pd.isna(value):
        return None

    val = str(value).strip().upper()

    # 1️⃣ MMI then numeric only → Very low intensity
    if re.match(r'^MMI\s*\d+$', val):
        return 'Very low intensity'

    # 2️⃣ MMI VIIABOUT → High intensity
    elif 'MMI' in val and 'VIIABOUT' in val:
        return 'High intensity'

    # 3️⃣ MMI VII + number → High intensity
    elif re.match(r'^MMI\s*VII\d+', val):
        return 'High intensity'

    # 4️⃣ MMI VIIIABOUT → Very high intensity
    elif 'MMI' in val and 'VIIIABOUT' in val:
        return 'Very high intensity'

    # 5️⃣ MMI VIII + number → Very high intensity
    elif re.match(r'^MMI\s*VIII\d+', val):
        return 'Very high intensity'

    # 6️⃣ MMI VI + number → High intensity
    elif re.match(r'^MMI\s*VI\d+', val):
        return 'High intensity'

    # 7️⃣ MMI V + number → Moderate intensity
    elif re.match(r'^MMI\s*V\d+', val):
        return 'Moderate intensity'

    # Already clean categories
    elif '100KM' in val:
        return 'Within 100 km'
    elif 'MODERATE' in val:
        return 'Moderate intensity'
    elif 'HIGH' in val:
        return 'High intensity'
    elif 'VERY LOW' in val:
        return 'Very low intensity'
    elif 'VERY HIGH' in val:
        return 'Very high intensity'

    return val  # fallback, if none matched


# Apply function
df['impact_category'] = df['impact_category'].apply(categorize_impact)

# Save back to CSV
df.to_csv("updated_impact_category.csv", index=False)
print("✅ impact_category column categorized and saved as updated_impact_zones.csv")


✅ impact_category column categorized and saved as updated_impact_zones.csv


In [10]:
import pandas as pd
import re

# Load your CSV
df = pd.read_csv("updated_impact_category.csv")

check = df['impact_category'].unique()

print(check)


['WITHIN 100 KM' 'Moderate intensity' 'High intensity'
 'Very low intensity' nan 'Very high intensity' 'MMI  ABOUT' 'MMI  VABOUT'
 'MMI  VIABOUT' 'MMI -' 'MMI  VIIFEW']


In [12]:
import pandas as pd
import re

# Load your file
df = pd.read_csv("updated_impact_category.csv")

def categorize_impact(value):
    if pd.isna(value):
        return None

    val = str(value).strip().upper()

    # ✅ Handle your new specific cases first
    if val in ['MMI  ABOUT', 'MMI -']:
        return 'Very low intensity'
    elif val == 'MMI  VABOUT':
        return 'Moderate intensity'
    elif val in ['MMI  VIABOUT', 'MMI  VIIFEW']:
        return 'High intensity'

    # ✅ Then general regex-based logic
    if re.match(r'^MMI\s*\d+$', val):
        return 'Very low intensity'
    elif 'MMI' in val and 'VIIABOUT' in val:
        return 'High intensity'
    elif re.match(r'^MMI\s*VII\d+', val):
        return 'High intensity'
    elif 'MMI' in val and 'VIIIABOUT' in val:
        return 'Very high intensity'
    elif re.match(r'^MMI\s*VIII\d+', val):
        return 'Very high intensity'
    elif re.match(r'^MMI\s*VI\d+', val):
        return 'High intensity'
    elif re.match(r'^MMI\s*V\d+', val):
        return 'Moderate intensity'
    elif '100KM' in val:
        return 'Within 100 km'
    elif 'MODERATE' in val:
        return 'Moderate intensity'
    elif 'HIGH' in val:
        return 'High intensity'
    elif 'VERY LOW' in val:
        return 'Very low intensity'
    elif 'VERY HIGH' in val:
        return 'Very high intensity'

    return val  # fallback if none matched


# Apply the function to the column
df['impact_category'] = df['impact_category'].apply(categorize_impact)

# Optional: Check final unique values
print("✅ Cleaned unique values:\n", df['impact_category'].unique())

# Save to new file
df.to_csv("cleaned_earthquake.csv", index=False)
print("✅ Saved cleaned data as updated_impact_category_cleaned.csv")


✅ Cleaned unique values:
 ['WITHIN 100 KM' 'Moderate intensity' 'High intensity'
 'Very low intensity' None]
✅ Saved cleaned data as updated_impact_category_cleaned.csv


In [13]:
import pandas as pd
import re

# Load your CSV
df = pd.read_csv("cleaned_earthquake.csv")

check = df['impact_category'].unique()

print(check)

['WITHIN 100 KM' 'Moderate intensity' 'High intensity'
 'Very low intensity' nan]


In [ ]:
## To add new column as disaster_event_id 

In [2]:
import pandas as pd

# change filenames as needed
INPUT = "Tropical-Cyclone-updated.csv"
OUTPUT = "Tropical-Cyclone-updated-01.csv"

# read all as strings to preserve formatting
df = pd.read_csv(INPUT, dtype=str)

# auto-detect column name
col = None
for c in ("disaster_id", "disaster_ID", "disasterId"):
    if c in df.columns:
        col = c
        break
if col is None:
    raise KeyError("Could not find column 'disaster_id' or 'disaster_ID' in the file.")

def collapse_second_suffix(s):
    s = "" if pd.isna(s) else str(s).strip()
    parts = s.split('_')
    # If there are 5 or more parts (e.g. ..._<date>_<seq>_<split>), remove only the final numeric suffix
    # This avoids removing the date or single suffix values
    if len(parts) >= 5 and parts[-1].isdigit():
        return '_'.join(parts[:-1])
    return s

# Create new column
df['disaster_event_id'] = df[col].apply(collapse_second_suffix)

# Quick checks
total = len(df)
changed = df[df[col] != df['disaster_event_id']]
print(f"Total rows: {total}")
print(f"Rows changed (last suffix removed): {len(changed)}")
if len(changed) > 0:
    print("\nSample changed rows (original -> new):")
    print(changed[[col, 'disaster_event_id']].head(50).to_string(index=False))

# Spot-check a few random rows so you can confirm unchanged ones
print("\nRandom sample to verify unchanged examples:")
print(df[[col, 'disaster_event_id']].sample(min(10, total), random_state=1).to_string(index=False))

# Save result
df.to_csv(OUTPUT, index=False)
print(f"\n✅ Saved updated file to: {OUTPUT}")


Total rows: 684
Rows changed (last suffix removed): 279

Sample changed rows (original -> new):
           disaster_id   disaster_event_id
   TC_UNK_20110726_6_1   TC_UNK_20110726_6
   TC_UNK_20110727_7_1   TC_UNK_20110727_7
   TC_CAN_20110830_9_1   TC_CAN_20110830_9
   TC_CAN_20110830_9_2   TC_CAN_20110830_9
   TC_CAN_20110830_9_3   TC_CAN_20110830_9
  TC_CHN_20110823_11_1  TC_CHN_20110823_11
  TC_DOM_20110820_13_1  TC_DOM_20110820_13
  TC_DOM_20110820_13_2  TC_DOM_20110820_13
  TC_DOM_20110820_13_3  TC_DOM_20110820_13
  TC_BMU_20110921_17_1  TC_BMU_20110921_17
  TC_BMU_20110921_17_2  TC_BMU_20110921_17
  TC_BMU_20110921_17_3  TC_BMU_20110921_17
  TC_BMU_20110921_17_4  TC_BMU_20110921_17
  TC_BMU_20110921_17_5  TC_BMU_20110921_17
  TC_BLZ_20111024_22_1  TC_BLZ_20111024_22
  TC_BLZ_20111024_22_2  TC_BLZ_20111024_22
  TC_REU_20120229_30_1  TC_REU_20120229_30
  TC_REU_20120229_30_2  TC_REU_20120229_30
  TC_IND_20121030_57_1  TC_IND_20121030_57
  TC_CUB_20121022_58_1  TC_CUB_20121022_58
 

In [1]:
import pandas as pd
import pycountry
import re

# Load dataset
df = pd.read_csv("Forest Fires.csv", dtype=str)

# Your country column
country_col = "country"

# Get a clean list of all country names and common short forms
country_list = [c.name.lower() for c in pycountry.countries]
# Add extra manual forms for common variants
extra_names = ["usa", "uk", "uae", "russia", "viet nam"]
country_list.extend(extra_names)

# Function to detect how many real countries exist in the cell
def count_real_countries(text):
    if pd.isna(text) or not isinstance(text, str):
        return 0

    # Split text into parts by comma, semicolon, or 'and'
    parts = re.split(r'[;,]| and ', text)
    parts = [p.strip().lower() for p in parts if p.strip()]

    count = 0
    for p in parts:
        # Check if any real country name appears in part
        for c in country_list:
            # Match full word
            if re.search(r'\b' + re.escape(c) + r'\b', p):
                count += 1
                break
    return count

# Apply to the dataframe
df["num_countries"] = df[country_col].apply(count_real_countries)

# Count how many rows have more than one country
multi_country_rows = df[df["num_countries"] > 1]
count_multi = len(multi_country_rows)

print(f"Total rows: {len(df)}")
print(f"Rows with more than one country: {count_multi}")

# Optional: show a few examples
print("\nExamples of rows with multiple countries:")
print(multi_country_rows[[country_col, "num_countries"]].head(10))


Total rows: 3055
Rows with more than one country: 86

Examples of rows with multiple countries:
                                               country  num_countries
25   Central African Republic, The Democratic Repub...              2
38   The Democratic Republic of Congo, Central Afri...              2
161                     Kazakhstan, Russian Federation              2
451                               Mozambique, Zimbabwe              2
503                               Mozambique, Zimbabwe              2
521                    Sudan, Central African Republic              2
524                                  Cameroon, Nigeria              2
526                                     Chad, Cameroon              2
544                                Benin, Burkina Faso              2
559                                Burkina Faso, Benin              2


In [22]:
pip install pycountry

   ---------------------------------------- 0.0/6.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/6.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/6.3 MB ? eta -:--:--
   - -------------------------------------- 0.3/6.3 MB ? eta -:--:--
   --- ------------------------------------ 0.5/6.3 MB 1.3 MB/s eta 0:00:05
   ---- ----------------------------------- 0.8/6.3 MB 1.5 MB/s eta 0:00:04
   ------ --------------------------------- 1.0/6.3 MB 1.5 MB/s eta 0:00:04
   --------- ------------------------------ 1.6/6.3 MB 1.8 MB/s eta 0:00:03
   ------------- -------------------------- 2.1/6.3 MB 1.8 MB/s eta 0:00:03
   ------------- -------------------------- 2.1/6.3 MB 1.8 MB/s eta 0:00:03
   -------------- ------------------------- 2.4/6.3 MB 1.5 MB/s eta 0:00:03
   -------------- ------------------------- 2.4/6.3 MB 1.5 MB/s eta 0:00:03
   -------------- ------------------------- 2.4/6.3 MB 1.5 MB/s eta 0:00:03
   ---------------- --------------------

In [2]:
import pandas as pd
import pycountry
import re

# Load dataset
df = pd.read_csv("Forest Fires.csv", dtype=str)

country_col = "country"

# Get all country names (lowercased)
country_list = [c.name.lower() for c in pycountry.countries]
# Add some extra common variants
extra_names = ["usa", "uk", "uae", "russia", "viet nam", "turkiye"]
country_list.extend(extra_names)

def extract_countries(text):
    """Return list of real countries found in text"""
    if pd.isna(text) or not isinstance(text, str):
        return []
    parts = re.split(r'[;,]| and ', text)
    parts = [p.strip().lower() for p in parts if p.strip()]

    found = []
    for p in parts:
        for c in country_list:
            if re.search(r'\b' + re.escape(c) + r'\b', p):
                found.append(c.title())
                break
    return list(set(found))  # remove duplicates

# Apply the function
df["country_list"] = df[country_col].apply(extract_countries)

# Expand rows having multiple countries
expanded_rows = []
for _, row in df.iterrows():
    countries = row["country_list"]
    if not countries:  # if empty, keep as is
        expanded_rows.append(row)
    else:
        for c in countries:
            new_row = row.copy()
            new_row[country_col] = c
            expanded_rows.append(new_row)

# Create a new DataFrame
df_expanded = pd.DataFrame(expanded_rows)

# Drop helper column
df_expanded = df_expanded.drop(columns=["country_list"])

# Save or print
df_expanded.to_csv("expanded_countries.csv", index=False)

print(f"✅ Original rows: {len(df)}")
print(f"✅ Expanded rows: {len(df_expanded)}")
print("Saved as 'expanded_countries.csv'")


✅ Original rows: 3055
✅ Expanded rows: 3141
Saved as 'expanded_countries.csv'


In [7]:
import pandas as pd
import pycountry

# Load your dataset
df = pd.read_csv("Flood_updated_01.csv")  # or pd.read_csv("your_file.csv")

# Function to get ISO3 from country name
def get_iso3(country_name):
    try:
        return pycountry.countries.lookup(country_name).alpha_3
    except:
        return None

# Fill missing ISO3 codes only where iso3 is NaN or empty
df['iso3'] = df.apply(
    lambda x: get_iso3(x['Country']) if pd.isna(x['iso3']) or x['iso3'] == '' else x['iso3'], axis=1
)

# Save updated file
df.to_csv("updated_file_with_iso3.csv", index=False)

print("✅ Missing ISO3 codes have been filled based on country names!")


✅ Missing ISO3 codes have been filled based on country names!


In [10]:
import pandas as pd
import re
import numpy as np

# Load dataset
df = pd.read_csv("updated_file_with_iso3.csv")  # or Excel if needed

def extract_population(value):
    if pd.isna(value) or not isinstance(value, str):
        return np.nan

    text = value.lower().strip()

    # Case 1: "few"
    if "few" in text:
        return "few"

    # Case 2: "thousand"
    thousand_match = re.search(r'(\d+(\.\d+)?)\s*thousand', text)
    if thousand_match:
        num = float(thousand_match.group(1)) * 1000
        return int(num)

    # ✅ Case 3: Extract valid numbers (ignore those near km)
    # Matches digits only when NOT followed by "km"
    numbers = re.findall(r'\b(\d+)\b(?!\s*km)', text)

    # Convert to int
    numbers = [int(n) for n in numbers if n.isdigit()]

    # Additional check: if number appears right before "km" → drop it
    cleaned_numbers = []
    for n in numbers:
        # pattern like '100km' or 'within 100km'
        if re.search(rf'{n}\s*km', text):
            continue
        cleaned_numbers.append(n)

    if len(cleaned_numbers) > 1:
        return sum(cleaned_numbers)
    elif len(cleaned_numbers) == 1:
        return cleaned_numbers[0]
    else:
        return np.nan

# Apply to dataset
df["Population_effected"] = df["Exposed Population"].apply(extract_population)

# Save
df.to_csv("updated_with_population_effected_02.csv", index=False)

print("✅ Population_effected column created successfully — distance numbers like '100km' excluded.")


✅ Population_effected column created successfully — distance numbers like '100km' excluded.


In [11]:
import pandas as pd
import re
import numpy as np

# Load your dataset
df = pd.read_csv("updated_with_population_effected_02.csv")

def extract_impact_category(value):
    if pd.isna(value) or not isinstance(value, str):
        return np.nan

    text = value.strip()

    # 1️⃣ 100km case
    if re.search(r'100\s*km', text, re.IGNORECASE):
        return "100km"

    # 2️⃣ Capture patterns like "MMI>=VII", "MMI&>=IV", "MMI VI", "MMI VIIIAbout", etc.
    # This regex covers all these variants:
    match = re.search(r'MMI\s*[&>=]*\s*[IVX]+', text, re.IGNORECASE)
    if match:
        return match.group(0).strip().replace("  ", " ")

    # 3️⃣ Handle case like "in MMI" (without number)
    if "in MMI" in text:
        return "MMI"

    # 4️⃣ No match found
    return np.nan


# Apply the function
df["impact_category"] = df["Exposed Population"].apply(extract_impact_category)

# Save updated dataset
df.to_csv("updated_with_impact_category.csv", index=False)

print("✅ impact_category column created successfully.")


✅ impact_category column created successfully.


In [12]:
import pandas as pd
import re

# Load your CSV
df = pd.read_csv("updated_with_impact_category.csv")

check = df['impact_category'].unique()

print(check)

['100km' 'MMI IV' 'MMI>=VII' 'MMI VI' 'MMI&>=IV' 'MMI V' nan 'MMI'
 'MMI VII' 'MMI III' 'MMI II' 'MMI I' 'MMI&>=V' 'MMI&>=II' 'MMI&>=III'
 'MMI&>=I']


In [13]:
import pandas as pd
import numpy as np

# === Step 1: Load your dataset ===
# Change filename below to your actual file name
file_path = "updated_with_impact_category.csv"  # or "your_file.csv"

# Automatically detect file type
if file_path.endswith('.xlsx'):
    df = pd.read_excel(file_path)
else:
    df = pd.read_csv(file_path)

# === Step 2: Define your mapping dictionary ===
impact_mapping = {
    '100km': 'Within 100km',
    'MMI IV': 'Light',
    'MMI>=VII': 'Very Strong',
    'MMI VI': 'Strong',
    'MMI&>=IV': 'Light',
    'MMI V': 'Moderate',
    'MMI': 'Not Felt',
    'MMI VII': 'Very Strong',
    'MMI III': 'Weak',
    'MMI II': 'Weak',
    'MMI I': 'Not Felt',
    'MMI&>=V': 'Moderate',
    'MMI&>=II': 'Weak',
    'MMI&>=III': 'Weak',
    'MMI&>=I': 'Not Felt',
    np.nan: 'None'
}

# === Step 3: Normalize text before mapping (to handle extra spaces etc.) ===
df['impact_category'] = df['impact_category'].astype(str).str.strip().replace({'nan': np.nan})

# === Step 4: Map impact_category to new label column ===
df['impact_label'] = df['impact_category'].map(impact_mapping).fillna('None')

# === Step 5: Save updated dataset ===
output_path = "updated_output.csv"
df.to_csv(output_path, index=False)

print(f"✅ File updated successfully and saved as: {output_path}")


✅ File updated successfully and saved as: updated_output.csv


In [14]:
import pandas as pd
import pycountry
import re

# Load dataset
df = pd.read_csv("eruption.csv", dtype=str)

# Your country column
country_col = "country"

# Get a clean list of all country names and common short forms
country_list = [c.name.lower() for c in pycountry.countries]
# Add extra manual forms for common variants
extra_names = ["usa", "uk", "uae", "russia", "viet nam"]
country_list.extend(extra_names)

# Function to detect how many real countries exist in the cell
def count_real_countries(text):
    if pd.isna(text) or not isinstance(text, str):
        return 0

    # Split text into parts by comma, semicolon, or 'and'
    parts = re.split(r'[;,]| and ', text)
    parts = [p.strip().lower() for p in parts if p.strip()]

    count = 0
    for p in parts:
        # Check if any real country name appears in part
        for c in country_list:
            # Match full word
            if re.search(r'\b' + re.escape(c) + r'\b', p):
                count += 1
                break
    return count

# Apply to the dataframe
df["num_countries"] = df[country_col].apply(count_real_countries)

# Count how many rows have more than one country
multi_country_rows = df[df["num_countries"] > 1]
count_multi = len(multi_country_rows)

print(f"Total rows: {len(df)}")
print(f"Rows with more than one country: {count_multi}")

# Optional: show a few examples
print("\nExamples of rows with multiple countries:")
print(multi_country_rows[[country_col, "num_countries"]].head(10))


Total rows: 186
Rows with more than one country: 0

Examples of rows with multiple countries:
Empty DataFrame
Columns: [country, num_countries]
Index: []


In [21]:
import pandas as pd
import re

# === STEP 1: Load your Excel file ===
file_path = 'eruption.csv'  # <-- change this to your actual file path
df = pd.read_csv(file_path)

# === STEP 2: Split the "name" column into two parts ===
def split_name_and_number(text):
    if pd.isna(text):
        return None, None
    text = str(text).strip()

    # Look for number inside parentheses or anywhere
    match = re.search(r'\(?(\d{4,})\)?', text)
    if match:
        number = match.group(1)
        name = re.sub(r'\(?\d{4,}\)?', '', text).strip()
        name = re.sub(r'\s*\(\s*\)\s*', '', name).strip()  # remove empty parentheses
        return name, number
    else:
        return text, None

# Apply the function to split the name column
df[['Name', 'Volcano Number']] = df['Name'].apply(lambda x: pd.Series(split_name_and_number(x)))

# === STEP 3: Save to new Excel file ===
output_file = 'volcano_data_split.csv'
df.to_csv(output_file, index=False)

print(f"✅ File saved as: {output_file}")


✅ File saved as: volcano_data_split.csv


In [29]:
file_path = 'volcano.csv'
df = pd.read_csv(file_path)
df_unique = df['Population Exposure Index PEI'].unique()
print(df_unique)

[nan  5.  3.  2.  4.  7.  6.]


In [33]:
import pandas as pd

# Read your Excel file
df = pd.read_csv("volcano.csv")

# --- Mapping for Volcano Explosivity Index (VEI) ---
vei_map = {
    0: "Non-explosive",
    1: "Gentle",
    2: "Explosive",
    3: "Severe",
    4: "Cataclysmic",
    5: "Paroxysmal",
    6: "Colossal",
    7: "Super-colossal",
    8: "Mega-colossal"
}

# --- Mapping for Population Exposure Index (PEI) ---
pei_map = {
    0: "Uninhabited / Remote",
    1: "Very low exposure",
    2: "Low exposure",
    3: "Moderate exposure",
    4: "High exposure",
    5: "Very high exposure",
    6: "Extremely high exposure",
    7: "Maximum exposure / Highly urbanized zone"
}

# Replace column names
df = df.rename(columns={
    "Max Volc. Explosivity Index VEI": "volcano_explosivity_index",
    "Population Exposure Index PEI": "population_exposure_index"
})

# Replace numeric values with descriptions
df["volcano_explosivity_index"] = df["volcano_explosivity_index"].map(vei_map)
df["population_exposure_index"] = df["population_exposure_index"].map(pei_map)

# Fill empty cells with "0"
df["volcano_explosivity_index"] = df["volcano_explosivity_index"].fillna("Non-explosive")
df["population_exposure_index"] = df["population_exposure_index"].fillna("Uninhabited / Remote")

# Save to a new Excel file
df.to_csv("updated_volcano_data_01.csv", index=False)

print("✅ Descriptions added successfully! File saved as 'updated_volcano_data.xlsx'")


✅ Descriptions added successfully! File saved as 'updated_volcano_data.xlsx'


In [34]:
import pandas as pd
import re

# Read your Excel file
df = pd.read_csv("updated_volcano_data_01.csv")

# Define a function to extract population
def extract_population(text):
    if pd.isna(text):
        return 0
    
    text = str(text).lower().strip()
    
    # If text mentions no people
    if "no people" in text or "none" in text:
        return 0
    
    # Regex to capture numbers before 'people'
    matches = re.findall(r'(\d[\d,]*)\s*people', text)
    total = 0
    
    for match in matches:
        # Remove commas and convert to int
        num = int(match.replace(',', ''))
        total += num

    # If text mentions 'thousand' or 'million', adjust scale
    if "thousand" in text and total == 0:
        match = re.search(r'(\d+)\s*thousand', text)
        if match:
            total += int(match.group(1)) * 1000
    elif "million" in text and total == 0:
        match = re.search(r'(\d+)\s*million', text)
        if match:
            total += int(match.group(1)) * 1_000_000

    return total

# Apply the function to the column
df["population_effected"] = df["Exposed Population 100km"].apply(extract_population)

# Save to new Excel file
df.to_csv("updated_population_effected.csv", index=False)

print("✅ Population extracted successfully! File saved as 'updated_population_effected.xlsx'")


✅ Population extracted successfully! File saved as 'updated_population_effected.xlsx'


In [1]:
import pandas as pd
import pycountry
import re
from fuzzywuzzy import process

# Load dataset
df = pd.read_csv("Flood.csv", dtype=str)

country_col = "country"

# Create list of valid country names (lowercase)
country_list = [c.name.lower() for c in pycountry.countries]
extra_names = ["usa", "uk", "uae", "russia", "viet nam"]
country_list.extend(extra_names)

# Function to clean and correct possible country misspellings
def correct_country_name(name):
    if not name or not isinstance(name, str):
        return None

    name = name.strip().lower()

    # Find the best fuzzy match among country_list
    match, score = process.extractOne(name, country_list)
    if score >= 90:  # high confidence threshold
        return match.title()
    else:
        return name.title()  # return original if no close match

# Function to detect and correct countries in each row
def count_and_correct_countries(text):
    if pd.isna(text) or not isinstance(text, str):
        return {"corrected": None, "count": 0}

    # Split text into parts by commas, semicolons, or 'and'
    parts = re.split(r'[;,]| and ', text)
    parts = [p.strip() for p in parts if p.strip()]

    corrected_countries = []
    for part in parts:
        lower_p = part.lower()

        # Fuzzy match only if something close to a country name
        match, score = process.extractOne(lower_p, country_list)
        if score >= 85:  # flexible threshold for small typos
            corrected = match.title()
            corrected_countries.append(corrected)

    return {
        "corrected": ", ".join(corrected_countries) if corrected_countries else text,
        "count": len(corrected_countries)
    }

# Apply correction + counting
results = df[country_col].apply(count_and_correct_countries)

df["corrected_country"] = results.apply(lambda x: x["corrected"])
df["num_countries"] = results.apply(lambda x: x["count"])

# Count rows with more than one real country
multi_country_rows = df[df["num_countries"] > 1]
count_multi = len(multi_country_rows)

print(f"✅ Total rows: {len(df)}")
print(f"✅ Rows with more than one country: {count_multi}")

# Save corrected dataset
df.to_csv("corrected_countries.csv", index=False)

print("💾 Saved corrected file as 'corrected_countries.csv'")
print("\nExamples of corrected rows:")
print(df[[country_col, "corrected_country", "num_countries"]].head(10))


✅ Total rows: 2441
✅ Rows with more than one country: 36
💾 Saved corrected file as 'corrected_countries.csv'

Examples of corrected rows:
                 country corrected_country  num_countries
0             Mozambique        Mozambique              1
1           Philippines        Philippines              1
2                 Angola            Angola              1
3  Southeastern Brazil              Brazil              1
4             Mozambique        Mozambique              1
5             Madagascar        Madagascar              1
6             Australia          Australia              1
7                    USA               Usa              1
8           Philippines        Philippines              1
9                    USA               Usa              1


In [3]:
import pandas as pd

# === Load dataset ===
df = pd.read_csv("corrected_countries.csv", dtype=str)

# Define column names
country_col = "Country"
id_col = "disaster_id"

# Ensure no NaN values in these columns
df[country_col] = df[country_col].fillna("")
df[id_col] = df[id_col].fillna("")

# === Split rows where a country cell has multiple countries ===
new_rows = []

for _, row in df.iterrows():
    # Split by comma, semicolon, or "and"
    countries = [c.strip() for c in str(row[country_col]).replace(" and ", ",").replace(";", ",").split(",") if c.strip()]

    # If only one country → keep as is
    if len(countries) == 1:
        new_rows.append(row)
    else:
        # If multiple countries → create new rows
        for i, country in enumerate(countries):
            new_row = row.copy()
            new_row[country_col] = country
            # Keep same ID for first, and add suffix for others
            if i == 0:
                new_row[id_col] = row[id_col]
            else:
                new_row[id_col] = f"{row[id_col]}_{i}"
            new_rows.append(new_row)

# === Create new DataFrame ===
df_expanded = pd.DataFrame(new_rows)

# === Save to CSV (optional) ===
df_expanded.to_csv("eruption_expanded.csv", index=False)

# === Print summary ===
print(f"Original rows: {len(df)}")
print(f"Expanded rows: {len(df_expanded)}")
print("\nExamples of expanded rows:")
print(df_expanded[[id_col, country_col]].head(10))


Original rows: 2441
Expanded rows: 2607

Examples of expanded rows:
          disaster_id      Country
0   FL_MOZ_20000126_2   Mozambique
1   FL_PHL_20000128_3  Philippines
2   FL_AGO_20000108_4       Angola
3   FL_UNK_20000101_5       Brazil
4   FL_MOZ_20000126_6   Mozambique
5   FL_MDG_20000217_7   Madagascar
6   FL_AUS_20000218_8    Australia
7   FL_USA_20000218_9          Usa
8  FL_PHL_20000128_10  Philippines
9  FL_USA_20000329_11          Usa


In [ ]:
## To seperate the countries from one cell if any cell contain more than one country

In [1]:
import pandas as pd
import pycountry
import re

# === Load dataset ===
df = pd.read_csv("Tropical Cyclone.csv", dtype=str)

# Define columns
country_col = "country"
id_col = "disaster_id"

# Prepare clean list of country names
country_list = [c.name.lower() for c in pycountry.countries]
# Add some known variants
country_list.extend([
    "usa", "uk", "russia", "viet nam", "uae", "south korea", "north korea"
])

# Function to extract valid countries from a text
def extract_countries(text):
    if pd.isna(text) or not isinstance(text, str):
        return []
    text_lower = text.lower()
    found = []

    # Check for each valid country in text
    for country in country_list:
        pattern = r'\b' + re.escape(country) + r'\b'
        if re.search(pattern, text_lower):
            found.append(country.title())

    # Remove duplicates and preserve order
    found_unique = list(dict.fromkeys(found))
    return found_unique

# Expand rows
new_rows = []

for _, row in df.iterrows():
    countries = extract_countries(row[country_col])
    
    if len(countries) == 0:
        # If no match found, keep original text
        row[country_col] = row[country_col]
        new_rows.append(row)
    elif len(countries) == 1:
        # Only one valid country → keep as is
        row[country_col] = countries[0]
        new_rows.append(row)
    else:
        # Multiple valid countries → create new rows
        for i, country in enumerate(countries):
            new_row = row.copy()
            new_row[country_col] = country
            if i == 0:
                new_row[id_col] = row[id_col]
            else:
                new_row[id_col] = f"{row[id_col]}_{i}"
            new_rows.append(new_row)

# Create new DataFrame
df_expanded = pd.DataFrame(new_rows)

# Save output
df_expanded.to_csv("Tropical-Cyclone-updated.csv", index=False)

# Print summary
print(f"Original rows: {len(df)}")
print(f"Expanded rows: {len(df_expanded)}")

print("\nExamples of expanded rows:")
print(df_expanded[[id_col, country_col]].head(15))


Original rows: 405
Expanded rows: 684

Examples of expanded rows:
           disaster_id         country
0    TC_UNK_20080428_2       Off-shore
1    TC_UNK_20110522_3           Japan
2    TC_UNK_20110728_4           Japan
3    TC_UNK_20110731_5       Off-shore
4    TC_UNK_20110726_6           China
4  TC_UNK_20110726_6_1     Philippines
5    TC_UNK_20110727_7          Mexico
5  TC_UNK_20110727_7_1   United States
6    TC_UNK_20110718_8          Mexico
7    TC_CAN_20110830_9          Canada
7  TC_CAN_20110830_9_1  United Kingdom
7  TC_CAN_20110830_9_2     Isle Of Man
7  TC_CAN_20110830_9_3         Ireland
8   TC_JPN_20110825_10           Japan
9   TC_CHN_20110823_11           China


In [9]:
import pandas as pd
import pycountry
from fuzzywuzzy import process

# === Load dataset ===
df = pd.read_csv("Flood_updated_01.csv", dtype=str)

# Ensure columns exist
if 'Country' not in df.columns or 'iso3' not in df.columns:
    raise ValueError("Columns 'Country' and 'iso3' must exist in your file.")

# Clean country names
df['Country'] = df['Country'].fillna('').str.strip()

# Build mapping of valid country names
country_dict = {c.name: c.alpha_3 for c in pycountry.countries}
country_names = list(country_dict.keys())

# Add manual short forms / variants
manual_map = {
    "usa": "USA",
    "us": "USA",
    "uk": "GBR",
    "russia": "RUS",
    "viet nam": "VNM",
    "south korea": "KOR",
    "north korea": "PRK",
    "uae": "ARE",
    "iran": "IRN",
    "laos": "LAO",
    "moldova": "MDA",
    "syria": "SYR",
    "tanzania": "TZA",
    "bolivia": "BOL",
    "venezuela": "VEN"
}

# Function to get ISO3 robustly
def get_iso3(country_name):
    if not country_name:
        return None
    name = country_name.strip()

    # Try direct lookup
    try:
        return pycountry.countries.lookup(name).alpha_3
    except:
        pass

    # Try manual map
    low = name.lower()
    if low in manual_map:
        return manual_map[low]

    # Try fuzzy match (for near names)
    match, score = process.extractOne(name, country_names)
    if score >= 85:
        return country_dict[match]

    return None

# Fill missing ISO3 codes only where iso3 is blank or NaN
df['iso3'] = df.apply(
    lambda x: get_iso3(x['Country']) if pd.isna(x['iso3']) or x['iso3'] == '' else x['iso3'],
    axis=1
)

# Save updated file
df.to_csv("Flood_updated_03.csv", index=False)

print("✅ Missing ISO3 codes filled successfully!")


✅ Missing ISO3 codes filled successfully!


In [4]:
import pandas as pd
import pycountry
from fuzzywuzzy import process
import unicodedata
import re
import ftfy

# ---------- CONFIG ----------
INPUT = "Tropical-Cyclone-updated-01.csv"    # change to your file
OUTPUT = "Tropical-Cyclone-updated-02.csv"
COUNTRY_COL = "country"
ISO_COL = "iso3"
FUZZY_THRESHOLD = 85             # lower if you want more aggressive fuzzy matches
# -----------------------------

# load
df = pd.read_csv(INPUT, dtype=str)
df[COUNTRY_COL] = df[COUNTRY_COL].fillna("").astype(str).str.strip()
df[ISO_COL] = df[ISO_COL].fillna("").astype(str).str.strip()

# build country name list and dicts
pyc = list(pycountry.countries)
name_to_alpha3 = {c.name.lower(): c.alpha_3 for c in pyc}
# include common_name and official_name if present
for c in pyc:
    if hasattr(c, "common_name"):
        name_to_alpha3[c.common_name.lower()] = c.alpha_3
    if hasattr(c, "official_name"):
        name_to_alpha3[c.official_name.lower()] = c.alpha_3

all_country_names = list(name_to_alpha3.keys())

# manual overrides for common variants / short forms
manual_map = {
    "usa": "USA",
    "u.s.a.": "USA",
    "united states": "USA",
    "united states of america": "USA",
    "uk": "GBR",
    "united kingdom": "GBR",
    "russia": "RUS",
    "korea, republic of": "KOR",      # "Korea, Republic of" variant
    "republic of korea": "KOR",
    "korea south": "KOR",
    "south korea": "KOR",
    "north korea": "PRK",
    "viet nam": "VNM",
    "vietnam": "VNM",
    "venezuela, bolivarian republic of": "VEN",
    "bolivia (plurinational state of)": "BOL",
    "tanzania, united republic of": "TZA",
    "lao people's democratic republic": "LAO",
    "iran (islamic republic of)": "IRN",
    "syria": "SYR",
    "taiwan": "TWN",
    "china": "CHN",
    "mainland china": "CHN",
    "brunei": "BRN",
    # add more variants you know
}

# helper: normalize & repair text
def normalize_text(s):
    if s is None:
        return ""
    s = str(s).strip()
    if not s:
        return ""
    # Repair mojibake (bad encoding) using ftfy
    try:
        s = ftfy.fix_text(s)
    except Exception:
        pass
    # remove invisible control characters
    s = "".join(ch for ch in s if unicodedata.category(ch)[0] != "C")
    # collapse whitespace
    s = re.sub(r'\s+', ' ', s)
    return s.strip()

# helper: attempt to resolve to iso3
def resolve_iso3(raw_name):
    if not raw_name:
        return None
    name = normalize_text(raw_name)
    low = name.lower().strip()

    # 1) manual map direct
    if low in manual_map:
        return manual_map[low]

    # 2) direct pycountry lookup (works with many variants)
    try:
        c = pycountry.countries.lookup(name)
        return c.alpha_3
    except Exception:
        pass

    # 3) exact normalized name in our name_to_alpha3
    if low in name_to_alpha3:
        return name_to_alpha3[low]

    # 4) fuzzy match against country names
    match, score = process.extractOne(low, all_country_names)
    if match and score >= FUZZY_THRESHOLD:
        return name_to_alpha3[match]

    # 5) attempt simpler heuristics: remove common prefixes/suffixes
    # e.g., "republic of x", "kingdom of x", "state of ..." -> try to extract core tokens
    heur = re.sub(r'^(the|republic of|kingdom of|state of|commonwealth of|federated states of)\s+', '', low)
    heur = re.sub(r',.*$', '', heur).strip()  # remove anything after comma
    if heur in name_to_alpha3:
        return name_to_alpha3[heur]
    match2, score2 = process.extractOne(heur, all_country_names)
    if match2 and score2 >= (FUZZY_THRESHOLD - 5):
        return name_to_alpha3[match2]

    return None

# Now apply to rows where iso3 is missing / blank
filled = 0
for idx, row in df.iterrows():
    current_iso = str(row.get(ISO_COL, "")).strip()
    if current_iso:   # skip rows already having iso3
        continue
    country_raw = row.get(COUNTRY_COL, "")
    iso = resolve_iso3(country_raw)
    if iso:
        df.at[idx, ISO_COL] = iso
        filled += 1

print(f"Filled {filled} missing iso3 values. Now summarizing remaining missing values...")

# Report remaining missing iso3
missing = df[df[ISO_COL].isnull() | (df[ISO_COL].str.strip() == "")]
if not missing.empty:
    print(f"Remaining rows without iso3: {len(missing)}. Showing unique country names that failed resolution:")
    print(missing[COUNTRY_COL].dropna().unique()[:200])  # show up to first 200 unique values
else:
    print("All iso3 resolved!")

# Save
df.to_csv(OUTPUT, index=False)
print(f"Saved result to {OUTPUT}")


Filled 17 missing iso3 values. Now summarizing remaining missing values...
Remaining rows without iso3: 156. Showing unique country names that failed resolution:
['Off-shore']
Saved result to Tropical-Cyclone-updated-02.csv


In [6]:
import pandas as pd
import ftfy

# Read your dataset (replace with your actual file path)
df = pd.read_csv("Flood_updated_04.csv")

# --- 1️⃣ Fix encoding issues like "TÃ¼rkiye" -> "Türkiye" ---
df["Country"] = df["Country"].apply(lambda x: ftfy.fix_text(str(x)) if pd.notna(x) else x)

# --- 2️⃣ Normalize long or reversed country names ---
country_replacements = {
    "Iran, Islamic Republic Of": "Islamic Republic of Iran",
    "Congo, The Democratic Republic Of The": "Democratic Republic of the Congo",
    "Tanzania, United Republic Of": "United Republic of Tanzania",
    "Venezuela, Bolivarian Republic Of": "Bolivarian Republic of Venezuela",
    "Korea, Democratic People's Republic Of": "Democratic People's Republic of Korea",
    "Korea, Republic Of": "Republic of Korea",
    "Moldova, Republic Of": "Republic of Moldova",
    "Lao People's Democratic Republic": "Lao People's Democratic Republic",  # sometimes miswritten
    "Syrian Arab Republic": "Syrian Arab Republic",
    "Palestine, State Of": "State of Palestine",
    "Micronesia, Federated States Of": "Federated States of Micronesia",
    "Bolivia, Plurinational State Of": "Plurinational State of Bolivia"
}

df["Country"] = df["Country"].replace(country_replacements)

# --- Optional: Fix capitalization (e.g., remove unwanted ALL CAPS) ---
df["Country"] = df["Country"].str.title()

# Save cleaned file
df.to_csv("Flood_updated_05.csv", index=False)

print("✅ Country column cleaned and saved as cleaned_countries.csv")


✅ Country column cleaned and saved as cleaned_countries.csv


In [7]:
file_path = 'Flood_updated_05.csv'
df = pd.read_csv(file_path)
df_unique = df['Country'].unique()
print(df_unique)

['Mozambique' 'Philippines' 'Angola' 'Brazil' 'Madagascar' 'Australia'
 'Usa' 'Argentina' 'Central African Republic' 'South Africa' 'Cambodia'
 'Zambia' 'Romania' 'China' 'Bangladesh' 'Viet Nam' 'Timor-Leste'
 'Colombia' 'Türkiye' 'Somalia' 'Guatemala' 'Chile' 'India' 'Thailand'
 'Russia' 'Japan' 'Republic Of Korea' 'Ethiopia'
 'Islamic Republic Of Iran' 'Cameroon' 'Mexico' 'Sri Lanka' 'Nigeria'
 'France' 'El Salvador' 'Indonesia' 'Ireland' 'Uk' 'Spain' 'Israel'
 'Algeria' 'Italy' 'Belize' 'Mali' 'Saudi Arabia'
 'Bolivarian Republic Of Venezuela' 'Canada' 'Taiwan, Province Of China'
 'United Republic Of Tanzania' 'Croatia' 'Plurinational State Of Bolivia'
 'Portugal' 'Malaysia' 'Kenya' 'Malawi' 'Peru' 'Germany' 'Ecuador'
 'Uruguay' 'Belarus' 'Poland' 'Haiti' 'Congo' 'Ghana' 'Nepal' 'East Timor'
 'Yemen' 'Pakistan' 'Nicaragua' 'Ivory Coast' 'Chad' 'Guinea' 'Sudan'
 'Uganda' 'Afghanistan' 'Rwanda' 'Greece' 'Cuba' 'Syrian Arab Republic'
 'Morocco' 'Serbia' 'Zimbabwe' 'Lithuania' 'New Zeal